# Materials-Aware Validation of Quantum Material Band-Gap Prediction

## Objective

Evaluate whether the enhanced Random Forest model can generalize
to chemically related materials that were not present in the training set.

The previous notebook used random train/test splitting and 5-fold
cross-validation. Here we investigate whether those results remain
consistent under a materials-aware validation strategy.

In [2]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [3]:
data_path = Path("../data/enhanced_material_descriptors.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (93902, 26)

Columns:
['num_elements', 'total_atoms', 'mean_atomic_number', 'min_atomic_number', 'max_atomic_number', 'mean_atomic_mass', 'min_atomic_mass', 'max_atomic_mass', 'mean_atomic_radius', 'min_atomic_radius', 'max_atomic_radius', 'crys', 'spg_number', 'mean_electronegativity', 'min_electronegativity', 'max_electronegativity', 'electronegativity_difference', 'mean_ionization_energy', 'mean_electron_affinity', 'mean_s_valence', 'mean_p_valence', 'mean_d_valence', 'mean_f_valence', 'mean_period', 'mean_group', 'target_bandgap']


In [4]:
X = df.drop(columns=["target_bandgap"])
y = df["target_bandgap"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (93902, 25)
y shape: (93902,)


In [5]:
# ============================================================
# LOAD ORIGINAL JARVIS IDENTIFIERS
# ============================================================

from jarvis.db.figshare import data

dft_3d = data("dft_3d")

original_df = pd.DataFrame(dft_3d)

print("Original dataset shape:", original_df.shape)

print("\nIdentifier columns:")
print(original_df[["jid", "formula", "spg_number", "crys"]].head())

Obtaining 3D dataset 94k ...
Reference:https://doi.org/10.1016/j.commatsci.2025.114063
Other versions:https://doi.org/10.6084/m9.figshare.6815699
Loading the zipfile...
Loading completed.
Original dataset shape: (93902, 64)

Identifier columns:
           jid   formula spg_number        crys
0  JVASP-90856  TiCuSiAs        129  tetragonal
1  JVASP-86097      DyB6        221       cubic
2  JVASP-64906   Be2OsRu        119  tetragonal
3  JVASP-98225       KBi         14  monoclinic
4     JVASP-10      VSe2        164    trigonal


In [6]:
print("Enhanced rows:", len(df))
print("Original rows:", len(original_df))

print(
    "Same number of rows:",
    len(df) == len(original_df)
)

Enhanced rows: 93902
Original rows: 93902
Same number of rows: True


In [7]:
comparison = pd.DataFrame({
    "saved_target": df["target_bandgap"].head(10).values,
    "original_target": original_df["optb88vdw_bandgap"].head(10).values
})

print(comparison)

   saved_target  original_target
0         0.000            0.000
1         0.000            0.000
2         0.000            0.000
3         0.472            0.472
4         0.000            0.000
5         0.000            0.000
6         0.000            0.000
7         0.000            0.000
8         0.000            0.000
9         0.689            0.689


In [8]:
# ============================================================
# ATTACH MATERIAL IDENTIFIERS
# ============================================================

df["jid"] = original_df["jid"].values
df["formula"] = original_df["formula"].values

print(df[["jid", "formula", "target_bandgap"]].head(10))

           jid    formula  target_bandgap
0  JVASP-90856   TiCuSiAs           0.000
1  JVASP-86097       DyB6           0.000
2  JVASP-64906    Be2OsRu           0.000
3  JVASP-98225        KBi           0.472
4     JVASP-10       VSe2           0.000
5  JVASP-14014     TbMnSi           0.000
6  JVASP-64664    Ba4NaBi           0.000
7  JVASP-22556     SrFeO3           0.000
8  JVASP-86726    LuNi4Sn           0.000
9  JVASP-28634  MoW3Se2S6           0.689


In [9]:
formula_counts = df["formula"].value_counts()

print("Unique formulas:", df["formula"].nunique())
print("Total materials:", len(df))

print("\nMost repeated formulas:")
print(formula_counts.head(20))

Unique formulas: 65395
Total materials: 93902

Most repeated formulas:
formula
Li7Mn2Co3O12    102
Li4MnCo2O7      102
SiO2             94
CdI2             61
C                56
VO2              49
LiVF4            41
LiMnPO4          40
VOF              40
TiO2             40
MnO2             36
BN               36
Li3MnCoO5        36
FeOF             35
Si               35
Li7Mn4CoO12      34
LiFeSiO4         33
VOF3             31
PbS              31
CoO2             31
Name: count, dtype: int64


In [10]:
duplicate_formula_groups = formula_counts[formula_counts > 1]

print(
    "\nNumber of formulas appearing more than once:",
    len(duplicate_formula_groups)
)

print(
    "Materials belonging to repeated formulas:",
    duplicate_formula_groups.sum()
)


Number of formulas appearing more than once: 14544
Materials belonging to repeated formulas: 43051


In [11]:
print("\nNumber of elements distribution:")
print(df["num_elements"].value_counts().sort_index())

print("\nFormula examples:")
print(df["formula"].sample(20, random_state=42).tolist())


Number of elements distribution:
num_elements
1      915
2    22648
3    54556
4    13664
5     1960
6      147
7       12
Name: count, dtype: int64

Formula examples:
['MgHg', 'CoMo', 'Ba4MgIr', 'Li4CO4', 'TbGa3', 'Ti2CoRe', 'MoWSe4', 'SrBe2Ni', 'Pr3Mg', 'Sr2FeCuPb2O6', 'BrKr', 'Co2SbO6', 'BaMg2Se', 'IrN2', 'Sc2Sb', 'TmGa2', 'NaTlHg', 'AlCr2Re', 'LiFeF4', 'LiBe2Os']


### create chemical-system groups

In [12]:
import re

def get_element_system(formula):
    elements = re.findall(r"[A-Z][a-z]?", formula)
    return "-".join(sorted(set(elements)))

df["element_system"] = df["formula"].apply(get_element_system)

print(df[["formula", "element_system"]].sample(20, random_state=42))

            formula element_system
36420          MgHg          Hg-Mg
53507          CoMo          Co-Mo
4195        Ba4MgIr       Ba-Ir-Mg
26147        Li4CO4         C-Li-O
55294         TbGa3          Ga-Tb
27595       Ti2CoRe       Co-Re-Ti
39728        MoWSe4        Mo-Se-W
10021       SrBe2Ni       Be-Ni-Sr
12944         Pr3Mg          Mg-Pr
82664  Sr2FeCuPb2O6  Cu-Fe-O-Pb-Sr
56582          BrKr          Br-Kr
49501       Co2SbO6        Co-O-Sb
78677       BaMg2Se       Ba-Mg-Se
70896          IrN2           Ir-N
21062         Sc2Sb          Sb-Sc
34499         TmGa2          Ga-Tm
75934        NaTlHg       Hg-Na-Tl
71269       AlCr2Re       Al-Cr-Re
51435        LiFeF4        F-Fe-Li
29370       LiBe2Os       Be-Li-Os


## inspect chemical-system groups

In [13]:
system_counts = df["element_system"].value_counts()

print("Unique chemical systems:", df["element_system"].nunique())

print("\nMost common chemical systems:")
print(system_counts.head(20))

print("\nChemical systems with only one material:",
      (system_counts == 1).sum())

print("\nChemical systems with more than one material:",
      (system_counts > 1).sum())

Unique chemical systems: 32707

Most common chemical systems:
element_system
Co-Li-Mn-O      407
F-O-V           251
F-Mn-O          222
F-Fe-O          206
F-Li-O-V        170
F-Li-Mn         147
F-Li-V          143
Co-Li-O         130
Li-Mn-O         127
Mo-S-Se-Te-W    115
F-Li-Mn-O       110
O-Si            101
Mo-S-Se-W        96
C-Cd-H-O         93
O-V              90
Fe-Li-O          88
Li-Mn-O-P        86
C-H-O            83
Co-F-O           81
Mo-Se-Te-W       80
Name: count, dtype: int64

Chemical systems with only one material: 17240

Chemical systems with more than one material: 15467


## prepare features and groups

In [14]:
feature_columns = [
    "num_elements",
    "total_atoms",
    "mean_atomic_number",
    "min_atomic_number",
    "max_atomic_number",
    "mean_atomic_mass",
    "min_atomic_mass",
    "max_atomic_mass",
    "mean_atomic_radius",
    "min_atomic_radius",
    "max_atomic_radius",
    "crys",
    "spg_number",
    "mean_electronegativity",
    "min_electronegativity",
    "max_electronegativity",
    "electronegativity_difference",
    "mean_ionization_energy",
    "mean_electron_affinity",
    "mean_s_valence",
    "mean_p_valence",
    "mean_d_valence",
    "mean_f_valence",
    "mean_period",
    "mean_group"
]

X = df[feature_columns]
y = df["target_bandgap"]

groups = df["element_system"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Groups:", groups.nunique())

X shape: (93902, 25)
y shape: (93902,)
Groups: 32707


In [15]:
group_sizes = groups.value_counts()

print("Smallest group size:", group_sizes.min())
print("Largest group size:", group_sizes.max())
print("Mean group size:", group_sizes.mean())
print("Median group size:", group_sizes.median())

print("\nGroup-size distribution:")
print(group_sizes.describe())

Smallest group size: 1
Largest group size: 407
Mean group size: 2.8710062066224356
Median group size: 1.0

Group-size distribution:
count    32707.000000
mean         2.871006
std          5.498205
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max        407.000000
Name: count, dtype: float64


In [16]:
numeric_features = [
    "num_elements",
    "total_atoms",
    "mean_atomic_number",
    "min_atomic_number",
    "max_atomic_number",
    "mean_atomic_mass",
    "min_atomic_mass",
    "max_atomic_mass",
    "mean_atomic_radius",
    "min_atomic_radius",
    "max_atomic_radius",
    "mean_electronegativity",
    "min_electronegativity",
    "max_electronegativity",
    "electronegativity_difference",
    "mean_ionization_energy",
    "mean_electron_affinity",
    "mean_s_valence",
    "mean_p_valence",
    "mean_d_valence",
    "mean_f_valence",
    "mean_period",
    "mean_group"
]

categorical_features = [
    "crys",
    "spg_number"
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

print("Pipeline created successfully.")

Pipeline created successfully.


In [17]:
from sklearn.model_selection import GroupKFold

group_kfold = GroupKFold(n_splits=5)

mae_scores = []
rmse_scores = []
r2_scores = []

for fold, (train_idx, val_idx) in enumerate(
    group_kfold.split(X, y, groups=groups), start=1
):
    
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]
    
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]
    
    pipeline.fit(X_train, y_train)
    
    predictions = pipeline.predict(X_val)
    
    mae = mean_absolute_error(y_val, predictions)
    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    r2 = r2_score(y_val, predictions)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)
    
    print(
        f"Fold {fold}: "
        f"MAE = {mae:.4f}, "
        f"RMSE = {rmse:.4f}, "
        f"R² = {r2:.4f}"
    )

Fold 1: MAE = 0.3122, RMSE = 0.6692, R² = 0.7469
Fold 2: MAE = 0.3317, RMSE = 0.7317, R² = 0.7085
Fold 3: MAE = 0.3326, RMSE = 0.7266, R² = 0.7152
Fold 4: MAE = 0.3047, RMSE = 0.6486, R² = 0.7376
Fold 5: MAE = 0.3137, RMSE = 0.6474, R² = 0.7247


In [18]:
from sklearn.model_selection import GroupKFold

group_kfold = GroupKFold(n_splits=5)

print("Checking chemical-system separation...\n")

for fold, (train_idx, val_idx) in enumerate(
    group_kfold.split(X, y, groups=groups), start=1
):
    
    train_groups = set(groups.iloc[train_idx])
    val_groups = set(groups.iloc[val_idx])
    
    overlap = train_groups.intersection(val_groups)
    
    print(
        f"Fold {fold}: "
        f"train materials = {len(train_idx)}, "
        f"validation materials = {len(val_idx)}, "
        f"overlapping chemical systems = {len(overlap)}"
    )

Checking chemical-system separation...

Fold 1: train materials = 75121, validation materials = 18781, overlapping chemical systems = 0
Fold 2: train materials = 75121, validation materials = 18781, overlapping chemical systems = 0
Fold 3: train materials = 75122, validation materials = 18780, overlapping chemical systems = 0
Fold 4: train materials = 75122, validation materials = 18780, overlapping chemical systems = 0
Fold 5: train materials = 75122, validation materials = 18780, overlapping chemical systems = 0


In [19]:
results = pd.DataFrame({
    "fold": [1, 2, 3, 4, 5],
    "MAE": mae_scores,
    "RMSE": rmse_scores,
    "R2": r2_scores
})

print(results)

print("\nMean ± Std:")
print(f"MAE  : {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"RMSE : {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
print(f"R²   : {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")

   fold       MAE      RMSE        R2
0     1  0.312235  0.669204  0.746928
1     2  0.331729  0.731702  0.708549
2     3  0.332633  0.726635  0.715229
3     4  0.304736  0.648577  0.737624
4     5  0.313689  0.647371  0.724668

Mean ± Std:
MAE  : 0.3190 ± 0.0112
RMSE : 0.6847 ± 0.0372
R²   : 0.7266 ± 0.0141


In [20]:
# ============================================================
# TUNED MATERIALS-AWARE VALIDATION
# Random Forest: min_samples_leaf = 1
# ============================================================

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import pandas as pd

print("Starting tuned chemical-system GroupKFold...")

Starting tuned chemical-system GroupKFold...


In [21]:
# ============================================================
# TUNED CHEMICAL-SYSTEM GROUPKFOLD
# CELL 1 — LOAD EVERYTHING
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ------------------------------------------------------------
# 1. Load enhanced descriptor dataset
# ------------------------------------------------------------

data_path = Path("../data/enhanced_material_descriptors.csv")

df = pd.read_csv(data_path)

print("Enhanced dataset loaded.")
print("Shape:", df.shape)


# ------------------------------------------------------------
# 2. Load original JARVIS data
# ------------------------------------------------------------

from jarvis.db.figshare import data

dft_3d = data("dft_3d")
original_df = pd.DataFrame(dft_3d)

print("Original JARVIS dataset:", original_df.shape)


# ------------------------------------------------------------
# 3. Attach formula and identifiers
# ------------------------------------------------------------

df["jid"] = original_df["jid"].values
df["formula"] = original_df["formula"].values


# ------------------------------------------------------------
# 4. Create chemical system
# ------------------------------------------------------------

import re

def get_element_system(formula):
    elements = re.findall(
        r"[A-Z][a-z]?",
        formula
    )

    return "-".join(
        sorted(set(elements))
    )


df["element_system"] = (
    df["formula"].apply(get_element_system)
)

print(
    "Unique chemical systems:",
    df["element_system"].nunique()
)


# ------------------------------------------------------------
# 5. Define final 25 features
# ------------------------------------------------------------

feature_columns = [
    "num_elements",
    "total_atoms",
    "mean_atomic_number",
    "min_atomic_number",
    "max_atomic_number",
    "mean_atomic_mass",
    "min_atomic_mass",
    "max_atomic_mass",
    "mean_atomic_radius",
    "min_atomic_radius",
    "max_atomic_radius",
    "mean_electronegativity",
    "min_electronegativity",
    "max_electronegativity",
    "electronegativity_difference",
    "mean_ionization_energy",
    "mean_electron_affinity",
    "mean_s_valence",
    "mean_p_valence",
    "mean_d_valence",
    "mean_f_valence",
    "mean_period",
    "mean_group",
    "crys",
    "spg_number"
]

# NOTE:
# There are actually 26 columns above because
# crys + spg_number are categorical features.
# This is the complete final feature set.


X = df[feature_columns]
y = df["target_bandgap"]
groups = df["element_system"]

print("\nFinal data prepared:")
print("X:", X.shape)
print("y:", y.shape)
print("Groups:", groups.nunique())

Enhanced dataset loaded.
Shape: (93902, 26)
Obtaining 3D dataset 94k ...
Reference:https://doi.org/10.1016/j.commatsci.2025.114063
Other versions:https://doi.org/10.6084/m9.figshare.6815699
Loading the zipfile...
Loading completed.
Original JARVIS dataset: (93902, 64)
Unique chemical systems: 32707

Final data prepared:
X: (93902, 25)
y: (93902,)
Groups: 32707


In [22]:
feature_columns = [
    "num_elements",
    "total_atoms",
    "mean_atomic_number",
    "min_atomic_number",
    "max_atomic_number",
    "mean_atomic_mass",
    "min_atomic_mass",
    "max_atomic_mass",
    "mean_atomic_radius",
    "min_atomic_radius",
    "max_atomic_radius",
    "mean_electronegativity",
    "min_electronegativity",
    "max_electronegativity",
    "electronegativity_difference",
    "mean_ionization_energy",
    "mean_electron_affinity",
    "mean_s_valence",
    "mean_p_valence",
    "mean_d_valence",
    "mean_f_valence",
    "mean_period",
    "mean_group",
    "crys",
    "spg_number"
]

X = df[feature_columns]
y = df["target_bandgap"]
groups = df["element_system"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Groups:", groups.nunique())

X shape: (93902, 25)
y shape: (93902,)
Groups: 32707


In [23]:
# ============================================================
# CELL 2 — PREPROCESSOR
# ============================================================

numeric_features = [
    col for col in feature_columns
    if col not in ["crys", "spg_number"]
]

categorical_features = [
    "crys",
    "spg_number"
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessor ready.")

Preprocessor ready.


In [26]:
# ============================================================
# Create the tuned Random Forest pipeline
# ============================================================

numeric_features = [
    "num_elements",
    "total_atoms",
    "mean_atomic_number",
    "min_atomic_number",
    "max_atomic_number",
    "mean_atomic_mass",
    "min_atomic_mass",
    "max_atomic_mass",
    "mean_atomic_radius",
    "min_atomic_radius",
    "max_atomic_radius",
    "mean_electronegativity",
    "min_electronegativity",
    "max_electronegativity",
    "electronegativity_difference",
    "mean_ionization_energy",
    "mean_electron_affinity",
    "mean_s_valence",
    "mean_p_valence",
    "mean_d_valence",
    "mean_f_valence",
    "mean_period",
    "mean_group"
]

categorical_features = [
    "crys",
    "spg_number"
]

# Numerical preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combined preprocessor
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Tuned Random Forest
tuned_rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=1.0,
    random_state=42,
    n_jobs=-1
)

# Complete pipeline
tuned_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", tuned_rf)
])

print("Tuned pipeline created successfully.")

Tuned pipeline created successfully.


In [27]:
# ============================================================
# CELL 4 — CHEMICAL-SYSTEM GROUPKFOLD
# ============================================================

group_kfold = GroupKFold(n_splits=5)

mae_scores = []
rmse_scores = []
r2_scores = []

fold_results = []

for fold, (train_idx, val_idx) in enumerate(
    group_kfold.split(X, y, groups=groups),
    start=1
):

    X_train_fold = X.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]

    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    print(f"\n{'='*50}")
    print(f"FOLD {fold}")
    print(f"{'='*50}")

    print("Training samples:", len(train_idx))
    print("Validation samples:", len(val_idx))

    # Verify no chemical-system leakage
    train_groups = set(groups.iloc[train_idx])
    val_groups = set(groups.iloc[val_idx])

    overlap = train_groups.intersection(val_groups)

    print("Overlapping chemical systems:", len(overlap))

    # Train
    tuned_pipeline.fit(
        X_train_fold,
        y_train_fold
    )

    # Predict
    predictions = tuned_pipeline.predict(
        X_val_fold
    )

    # Metrics
    mae = mean_absolute_error(
        y_val_fold,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_val_fold,
            predictions
        )
    )

    r2 = r2_score(
        y_val_fold,
        predictions
    )

    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)

    fold_results.append({
        "fold": fold,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

    print(f"MAE  : {mae:.4f} eV")
    print(f"RMSE : {rmse:.4f} eV")
    print(f"R²   : {r2:.4f}")


FOLD 1
Training samples: 75121
Validation samples: 18781
Overlapping chemical systems: 0
MAE  : 0.3033 eV
RMSE : 0.6566 eV
R²   : 0.7563

FOLD 2
Training samples: 75121
Validation samples: 18781
Overlapping chemical systems: 0
MAE  : 0.3243 eV
RMSE : 0.7248 eV
R²   : 0.7140

FOLD 3
Training samples: 75122
Validation samples: 18780
Overlapping chemical systems: 0
MAE  : 0.3243 eV
RMSE : 0.7218 eV
R²   : 0.7190

FOLD 4
Training samples: 75122
Validation samples: 18780
Overlapping chemical systems: 0
MAE  : 0.2967 eV
RMSE : 0.6416 eV
R²   : 0.7433

FOLD 5
Training samples: 75122
Validation samples: 18780
Overlapping chemical systems: 0
MAE  : 0.3061 eV
RMSE : 0.6415 eV
R²   : 0.7296


In [28]:
# ============================================================
# CELL 5 — FINAL GROUPKFOLD RESULTS
# ============================================================

group_results = pd.DataFrame(fold_results)

print("\n")
print("=" * 60)
print("TUNED CHEMICAL-SYSTEM GROUPKFOLD RESULTS")
print("=" * 60)

print(group_results.to_string(index=False))

mean_mae = np.mean(mae_scores)
std_mae = np.std(mae_scores)

mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

mean_r2 = np.mean(r2_scores)
std_r2 = np.std(r2_scores)

print("\nMean ± Standard Deviation")
print("-" * 40)

print(f"MAE  : {mean_mae:.4f} ± {std_mae:.4f} eV")
print(f"RMSE : {mean_rmse:.4f} ± {std_rmse:.4f} eV")
print(f"R²   : {mean_r2:.4f} ± {std_r2:.4f}")



TUNED CHEMICAL-SYSTEM GROUPKFOLD RESULTS
 fold      MAE     RMSE       R2
    1 0.303279 0.656649 0.756334
    2 0.324254 0.724795 0.714025
    3 0.324336 0.721804 0.719003
    4 0.296656 0.641560 0.743270
    5 0.306108 0.641545 0.729602

Mean ± Standard Deviation
----------------------------------------
MAE  : 0.3109 ± 0.0113 eV
RMSE : 0.6773 ± 0.0380 eV
R²   : 0.7324 ± 0.0156


In [29]:
# ============================================================
# CELL 6 — SAVE RESULTS
# ============================================================

results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

group_results.to_csv(
    results_dir / "tuned_groupkfold_results.csv",
    index=False
)

summary = {
    "model": "RandomForestRegressor",
    "n_estimators": 200,
    "min_samples_leaf": 1,
    "max_depth": None,
    "min_samples_split": 2,
    "max_features": 1.0,
    "grouping": "element_system",
    "n_splits": 5,
    "MAE_mean_eV": mean_mae,
    "MAE_std_eV": std_mae,
    "RMSE_mean_eV": mean_rmse,
    "RMSE_std_eV": std_rmse,
    "R2_mean": mean_r2,
    "R2_std": std_r2
}

import json

with open(
    results_dir / "tuned_groupkfold_summary.json",
    "w"
) as f:
    json.dump(summary, f, indent=4)

print("Results saved successfully.")

Results saved successfully.
